# 00 Environment and Dataset Validation

Use this notebook to confirm the runtime, package imports, Colab setup expectations, raw-data path, and expected table availability before moving to EDA.

This notebook stays manual by design. It does not copy or download data automatically.
Standard Phase 0 artifacts are saved under `data/processed/phase0_environment/`.


In [ ]:
from datetime import datetime, timezone
from pathlib import Path
import platform
import sys

try:
    import google.colab  # type: ignore  # noqa: F401
    RUNTIME = "colab"
except ImportError:
    RUNTIME = "local"

root = next(
    (candidate for candidate in [Path.cwd(), *Path.cwd().parents] if (candidate / "pyproject.toml").exists()),
    Path.cwd(),
)
src = root / "src"
if str(src) not in sys.path:
    sys.path.insert(0, str(src))

from credit_visable.utils.paths import get_paths

paths = get_paths()
DEFAULT_RAW_DATA_DIR = paths.data_raw
PHASE_0_OUTPUT_ROOT = paths.data_processed / "phase0_environment"
COLAB_INSTALL_COMMANDS = [
    "pip install -r requirements.txt",
    "pip install -e .",
    "pip install xgboost shap",
]
SESSION_STARTED_AT = datetime.now(timezone.utc).isoformat()

print(f"Runtime: {RUNTIME}")
print(f"Python version: {platform.python_version()}")
print(f"Current working directory: {Path.cwd()}")
print(f"Resolved project root: {paths.root}")
print(f"Default raw data directory: {DEFAULT_RAW_DATA_DIR}")
print(f"Phase 0 output root: {PHASE_0_OUTPUT_ROOT}")
print("Recommended Colab install commands:")
for command in COLAB_INSTALL_COMMANDS:
    print(f"- {command}")


In [ ]:
import importlib
import json

import pandas as pd
from IPython.display import display

module_specs = [
    {"module": "numpy", "required_for": "phase0_core"},
    {"module": "pandas", "required_for": "phase0_core"},
    {"module": "sklearn", "required_for": "phase2_phase4"},
    {"module": "matplotlib", "required_for": "phase1_visuals"},
    {"module": "seaborn", "required_for": "phase1_visuals"},
    {"module": "yaml", "required_for": "phase0_config"},
    {"module": "jupyter", "required_for": "notebook_runtime"},
    {"module": "xgboost", "required_for": "phase4_advanced_modeling"},
    {"module": "shap", "required_for": "phase5_explainability"},
]
rows = []
for spec in module_specs:
    module_name = spec["module"]
    try:
        importlib.import_module(module_name)
        rows.append(
            {
                "module": module_name,
                "required_for": spec["required_for"],
                "status": "ok",
                "detail": "import succeeded",
            }
        )
    except Exception as exc:
        rows.append(
            {
                "module": module_name,
                "required_for": spec["required_for"],
                "status": "failed",
                "detail": f"{exc.__class__.__name__}: {exc}",
            }
        )

import_status = pd.DataFrame(rows)
display(import_status)

failed_modules = import_status.loc[import_status["status"] != "ok", "module"].tolist()
if failed_modules:
    print("Phase 0 is blocked until the missing imports are fixed.")
    print(f"Missing or failing imports: {failed_modules}")
    print("Recommended recovery in Colab:")
    print("- pip install -r requirements.txt")
    print("- pip install -e .")
    print("- pip install xgboost shap")
else:
    print("All required Phase 0 / Colab runtime imports are available.")


In [ ]:
from credit_visable.config import load_settings

settings = load_settings()

# Edit this variable by hand in Colab when your Drive path differs from the repo default.
RAW_DATA_DIR = DEFAULT_RAW_DATA_DIR
# Colab example:
# RAW_DATA_DIR = Path("/content/drive/MyDrive/home-credit/data/raw")

print(f"Using raw data directory: {RAW_DATA_DIR}")
print(f"Configured target column: {settings.target_column}")
print(f"Configured customer key: {settings.id_column}")
print("Phase 0 rule: treat RAW_DATA_DIR as a run-time input, not a committed config change.")


In [ ]:
from credit_visable.data import summarize_table_availability

table_summary = summarize_table_availability(data_dir=RAW_DATA_DIR, settings=settings)
display(table_summary)

available_count = int(table_summary["available"].sum())
expected_count = int(len(table_summary))
print(f"Available tables: {available_count}/{expected_count}")

if available_count == 0:
    print("No expected raw tables were found in the selected raw-data directory.")
    print("Manual next steps:")
    print("- Copy the expected CSV files into data/raw/")
    print(' - Colab path override: RAW_DATA_DIR = Path("/content/drive/MyDrive/home-credit/data/raw")')
elif not bool(table_summary.loc[table_summary["table_name"] == "application_train", "available"].item()):
    print("application_train.csv is still missing, so do not continue to EDA yet.")
    print("Manual next steps:")
    print("- Copy application_train.csv into data/raw/")
    print(' - Colab path override: RAW_DATA_DIR = Path("/content/drive/MyDrive/home-credit/data/raw")')
else:
    print("application_train.csv is available. Partial table coverage is acceptable for Phase 0.")


In [ ]:
from credit_visable.data import load_application_train

application_train_available = bool(
    table_summary.loc[table_summary["table_name"] == "application_train", "available"].item()
)

if application_train_available:
    application_train_preview = load_application_train(data_dir=RAW_DATA_DIR, nrows=5)
    display(application_train_preview)
    print("Phase 0 is complete. Continue to notebooks/01_eda.ipynb once this preview looks correct.")
else:
    application_train_preview = pd.DataFrame()
    print("Phase 0 stop: application_train.csv must exist before you move to notebooks/01_eda.ipynb.")

PHASE_0_OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
table_availability_path = PHASE_0_OUTPUT_ROOT / "table_availability.csv"
import_status_path = PHASE_0_OUTPUT_ROOT / "import_status.csv"
runtime_summary_path = PHASE_0_OUTPUT_ROOT / "runtime_summary.json"

table_summary.to_csv(table_availability_path, index=False)
import_status.to_csv(import_status_path, index=False)

runtime_summary = {
    "session_started_at_utc": SESSION_STARTED_AT,
    "runtime": RUNTIME,
    "python_version": platform.python_version(),
    "project_root": str(paths.root),
    "raw_data_dir": str(RAW_DATA_DIR),
    "default_raw_data_dir": str(DEFAULT_RAW_DATA_DIR),
    "phase0_output_root": str(PHASE_0_OUTPUT_ROOT),
    "expected_table_count": expected_count,
    "available_table_count": available_count,
    "application_train_available": application_train_available,
    "import_status_ok": bool(import_status["status"].eq("ok").all()),
    "recommended_colab_install_commands": COLAB_INSTALL_COMMANDS,
}
runtime_summary_path.write_text(
    json.dumps(runtime_summary, indent=2, ensure_ascii=False),
    encoding="utf-8",
)

phase0_artifact_manifest = pd.DataFrame(
    [
        {"artifact": "table_availability", "path": str(table_availability_path)},
        {"artifact": "import_status", "path": str(import_status_path)},
        {"artifact": "runtime_summary", "path": str(runtime_summary_path)},
    ]
)
display(phase0_artifact_manifest)
